In [30]:
%pip install azure-storage-blob

Note: you may need to restart the kernel to use updated packages.


In [31]:
import os
import io
import os
import pandas as pd
from azure.storage.blob import BlobServiceClient
import numpy as np
import urllib.parse
import requests
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Carregar variáveis do .env
load_dotenv()

db_user = os.getenv('DB_USER')
db_pass = os.getenv('DB_PASS')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

senha_codificada = urllib.parse.quote_plus(db_pass)
string_conexao = f"postgresql+psycopg2://{db_user}:{senha_codificada}@{db_host}:{db_port}/{db_name}"
engine = create_engine(string_conexao)
print("Configuração carregada!")

Configuração carregada!


In [32]:
url_api = "https://raw.githubusercontent.com/maurobaptista/avaliacoes_produtos.json/main/avaliacoes_produtos.json"
resposta = requests.get(url_api)

if resposta.status_code == 200:
    df_avaliacoes = pd.json_normalize(resposta.json())
    # Limpeza dos nomes das colunas
    df_avaliacoes.columns = [col.replace('detalhes_avaliacao.', '') for col in df_avaliacoes.columns]
    print(f"Total de registos carregados: {len(df_avaliacoes)}")
else:
    print("Erro ao aceder à API.")

Total de registos carregados: 8


In [33]:
print(df_avaliacoes)

   id_pedido_fk  nota_estrelas  \
0          1001              5   
1          1002              3   
2          1003              4   
3          1005              1   
4          1006              5   
5          1007              2   
6          1008              5   
7          1009              4   

                                    comentario_texto        data_avaliacao  \
0    Excelente café, entrega muito rápida em Luanda.  2023-10-18T14:30:00Z   
1  A máquina chegou com um pequeno risco na lateral.  2023-10-20T09:15:00Z   
2                                                NaN  2023-10-22T11:00:00Z   
3  Péssimo serviço. O pedido atrasou duas semanas...  2023-11-05T16:45:00Z   
4  Os filtros de papel são de ótima qualidade, vo...  2023-11-10T10:20:00Z   
5  Sabor do café um pouco ácido para o meu gosto,...  2023-11-12T08:05:00Z   
6                                                NaN  2023-11-15T18:30:00Z   
7  Bom produto, mas o custo de frete para o Brasi...  2023-11-18T13:4

In [34]:
df_avaliacoes.head()

,id_pedido_fk,nota_estrelas,comentario_texto,data_avaliacao,compra_verificada
0,1001,5,"Excelente café, entrega muito rápida em Luanda.",2023-10-18T14:30:00Z,True
1,1002,3,A máquina chegou com um pequeno risco na lateral.,2023-10-20T09:15:00Z,True
2,1003,4,NaN,2023-10-22T11:00:00Z,True
3,1005,1,Péssimo serviço. O pedido atrasou duas semanas...,2023-11-05T16:45:00Z,True
4,1006,5,"Os filtros de papel são de ótima qualidade, vo...",2023-11-10T10:20:00Z,True


In [35]:
df_avaliacoes.tail()

,id_pedido_fk,nota_estrelas,comentario_texto,data_avaliacao,compra_verificada
3,1005,1,Péssimo serviço. O pedido atrasou duas semanas...,2023-11-05T16:45:00Z,True
4,1006,5,"Os filtros de papel são de ótima qualidade, vo...",2023-11-10T10:20:00Z,True
5,1007,2,"Sabor do café um pouco ácido para o meu gosto,...",2023-11-12T08:05:00Z,False
6,1008,5,NaN,2023-11-15T18:30:00Z,True
7,1009,4,"Bom produto, mas o custo de frete para o Brasi...",2023-11-18T13:40:00Z,True


In [36]:
df_avaliacoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_pedido_fk       8 non-null      int64
 1   nota_estrelas      8 non-null      int64
 2   comentario_texto   6 non-null      str  
 3   data_avaliacao     8 non-null      str  
 4   compra_verificada  8 non-null      bool 
dtypes: bool(1), int64(2), str(2)
memory usage: 396.0 bytes


In [37]:
df_avaliacoes.shape

(8, 5)

In [38]:
df_avaliacoes.columns

Index(['id_pedido_fk', 'nota_estrelas', 'comentario_texto', 'data_avaliacao',
       'compra_verificada'],
      dtype='str')

In [39]:

# Criar uma cópia de trabalho para não alterar os dados brutos
df_limpo = df_avaliacoes.copy()

# 1. Tratamento de Tipos e Datas
df_limpo['data_avaliacao'] = pd.to_datetime(df_limpo['data_avaliacao'])
df_limpo['id_pedido_fk'] = df_limpo['id_pedido_fk'].astype(int)
df_limpo['nota_estrelas'] = df_limpo['nota_estrelas'].astype(int)
df_limpo['compra_verificada'] = df_limpo['compra_verificada'].astype(bool)

# 2. Tratamento de Textos e Nulos
df_limpo['comentario_texto'] = df_limpo['comentario_texto'].str.strip()
df_limpo['tem_comentario'] = df_limpo['comentario_texto'].notna()
df_limpo['comentario_tratado'] = df_limpo['comentario_texto'].fillna('Sem comentário')

# 3. Engenharia de Atributos (NPS / Categoria de Satisfação)
condicoes = [
    df_limpo['nota_estrelas'] <= 2,
    df_limpo['nota_estrelas'] == 3,
    df_limpo['nota_estrelas'] >= 4
]
categorias = ['Detrator', 'Neutro', 'Promotor']
df_limpo['classe_nps'] = np.select(condicoes, categorias, default='Indefinido')

# 4. Extração de Componentes de Data
df_limpo['ano_mes'] = df_limpo['data_avaliacao'].dt.to_period('M').astype(str)

# 5. Visualizar o resultado final
df_limpo[['id_pedido_fk', 'nota_estrelas', 'classe_nps', 'tem_comentario', 'ano_mes', 'comentario_tratado']]

C:\Users\mauro_szbxxcy\AppData\Local\Temp\ipykernel_11580\580881164.py:25: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_limpo['ano_mes'] = df_limpo['data_avaliacao'].dt.to_period('M').astype(str)


,id_pedido_fk,nota_estrelas,classe_nps,tem_comentario,ano_mes,comentario_tratado
0,1001,5,Promotor,True,2023-10,"Excelente café, entrega muito rápida em Luanda."
1,1002,3,Neutro,True,2023-10,A máquina chegou com um pequeno risco na lateral.
2,1003,4,Promotor,False,2023-10,Sem comentário
3,1005,1,Detrator,True,2023-11,Péssimo serviço. O pedido atrasou duas semanas...
4,1006,5,Promotor,True,2023-11,"Os filtros de papel são de ótima qualidade, vo..."
5,1007,2,Detrator,True,2023-11,"Sabor do café um pouco ácido para o meu gosto,..."
6,1008,5,Promotor,False,2023-11,Sem comentário
7,1009,4,Promotor,True,2023-11,"Bom produto, mas o custo de frete para o Brasi..."


In [40]:

# 1. Forçar a leitura do .env atualizado do disco
load_dotenv(override=True)

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
nome_contentor = "bronze"
nome_ficheiro = "vendas.csv"

# Verificação de segurança antes de conectar
print("Início da string em uso:", str(connection_string)[:45])

# 2. Conectar ao serviço da Azure
blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service_client.get_container_client(nome_contentor)
blob_client = container_client.get_blob_client(nome_ficheiro)

print(f"A descarregar {nome_ficheiro} da camada Bronze na Azure...")

# 3. Leitura direta para a memória RAM
stream_bytes = blob_client.download_blob().readall()
df_vendas = pd.read_csv(io.BytesIO(stream_bytes))

print("✅ Ficheiro extraído com sucesso da Azure!")
df_vendas.head()

Início da string em uso: DefaultEndpointsProtocol=https;AccountName=ad
A descarregar vendas.csv da camada Bronze na Azure...
✅ Ficheiro extraído com sucesso da Azure!


,id_pedido,cliente,pais,valor_total,status_entrega,data_pedido
0,1001,João Silva,Angola,150.5,Entregue,2023-10-15
1,1002,Maria Costa,Portugal,89.9,Entregue,2023-10-16
2,1003,Carlos Dias,Angola,45.0,Entregue,2023-10-18
3,1004,Ana Paula,Brasil,210.0,Cancelado,2023-10-19
4,1005,Pedro Ramos,Portugal,120.3,Entregue,2023-10-25


In [41]:
# Cruzamento relacional mantendo todas as vendas
df_silver = pd.merge(
    df_vendas,
    df_limpo,
    how='left',
    left_on='id_pedido',
    right_on='id_pedido_fk'
)

# Selecionar e ordenar as colunas da camada Silver
colunas_finais = [
    'id_pedido',
    'cliente',
    'pais',
    'valor_total',
    'status_entrega',
    'data_pedido',
    'nota_estrelas',
    'classe_nps',
    'tem_comentario',
    'comentario_tratado'
]

df_silver = df_silver[colunas_finais]
df_silver

,id_pedido,cliente,pais,valor_total,status_entrega,data_pedido,nota_estrelas,classe_nps,tem_comentario,comentario_tratado
0,1001,João Silva,Angola,150.5,Entregue,2023-10-15,5.0,Promotor,True,"Excelente café, entrega muito rápida em Luanda."
1,1002,Maria Costa,Portugal,89.9,Entregue,2023-10-16,3.0,Neutro,True,A máquina chegou com um pequeno risco na lateral.
2,1003,Carlos Dias,Angola,45.0,Entregue,2023-10-18,4.0,Promotor,False,Sem comentário
3,1004,Ana Paula,Brasil,210.0,Cancelado,2023-10-19,NaN,NaN,NaN,NaN
4,1005,Pedro Ramos,Portugal,120.3,Entregue,2023-10-25,1.0,Detrator,True,Péssimo serviço. O pedido atrasou duas semanas...
5,1006,Helena Cruz,Angola,35.2,Entregue,2023-11-01,5.0,Promotor,True,"Os filtros de papel são de ótima qualidade, vo..."
6,1007,Rui Santos,Portugal,95.0,Entregue,2023-11-02,2.0,Detrator,True,"Sabor do café um pouco ácido para o meu gosto,..."
7,1008,Teresa Lima,Brasil,310.4,Entregue,2023-11-05,5.0,Promotor,False,Sem comentário
8,1009,Mário Nunes,Brasil,55.0,Entregue,2023-11-10,4.0,Promotor,True,"Bom produto, mas o custo de frete para o Brasi..."
9,1010,Luís Gomes,Angola,18.0,Em Trânsito,2023-11-12,NaN,NaN,NaN,NaN


In [42]:
# Envio para o PostgreSQL (substitui a tabela se já existir)
df_silver.to_sql(
    name='silver_vendas_avaliacoes',
    con=engine,
    if_exists='replace',
    index=False
)

print("🚀 Camada Silver carregada com sucesso no PostgreSQL!")

🚀 Camada Silver carregada com sucesso no PostgreSQL!


In [43]:
# ==========================================
# CAMADA GOLD - CRUZAMENTO FINAL NO PYTHON
# ==========================================
print("A extrair dados transacionais para a Camada Gold...")

# 1. Ler as tabelas transacionais do PostgreSQL para DataFrames Pandas
df_pedidos = pd.read_sql("SELECT * FROM pedidos", engine)
df_clientes = pd.read_sql("SELECT * FROM clientes", engine)
df_departamentos = pd.read_sql("SELECT * FROM departamentos", engine)

# 2. Construir a Tabela Gold (Desnormalizada)
# Cruzar Pedidos com Clientes
df_gold = pd.merge(df_pedidos, df_clientes, on='id_cliente', how='left')

# Cruzar o resultado com Departamentos
df_gold = pd.merge(df_gold, df_departamentos, on='id_dep', how='left')

# Cruzar finalmente com a nossa tabela Silver (que já tem dados do Azure e da API)
# Atenção às chaves: as tabelas cruzam através do 'id_pedido'
df_gold = pd.merge(df_gold, df_silver, on='id_pedido', how='left')

# 3. Limpeza Final (Remover colunas duplicadas ou desnecessárias geradas no join)
colunas_para_remover = ['id_cliente', 'id_dep']
df_gold = df_gold.drop(columns=colunas_para_remover)

# 4. Enviar a Camada Gold final para o PostgreSQL
df_gold.to_sql(
    name='gold_vendas_consolidadas',
    con=engine,
    if_exists='replace',
    index=False
)

print("🏆 Camada Gold construída e carregada com sucesso no PostgreSQL!")

A extrair dados transacionais para a Camada Gold...
🏆 Camada Gold construída e carregada com sucesso no PostgreSQL!
